In [2]:
from pymongo import MongoClient, errors
from pymongo.read_concern import ReadConcern
import threading
import time

In [3]:
uri = "mongodb://localhost:27017/?replicaSet=rs0"
client = MongoClient(uri)
db = client.lab1
coll = db.test_coll

In [4]:
coll.drop()
coll.insert_one({"_id": 1, "balance": 100, "status": "active"})

InsertOneResult(1, acknowledged=True)

In [5]:
def demo_dirty_read():
    print("\n--- Проверка Dirty Read (Грязное чтение) ---")
    
    def transaction_a():
        with client.start_session() as session:
            with session.start_transaction():
                print("[A] Начало транзакции. Меняю balance на 500...")
                coll.update_one({"_id": 1}, {"$set": {"balance": 500}}, session=session)
                val = coll.find_one({"_id": 1}, session=session)
                print(f"[A] Чтение данных: balance = {val['balance']}")
                time.sleep(3)
                print("[A] Отмена транзакции (Abort)...")
                session.abort_transaction()

    def transaction_b():
        time.sleep(1)
        val = coll.find_one({"_id": 1})
        print(f"[B] Чтение данных: balance = {val['balance']}")
        if val['balance'] == 500:
            print("[B] ОШИБКА: Вижу незафиксированные данные (Dirty Read!)")
        else:
            print("[B] УСПЕХ: Грязное чтение невозможно. База вернула старое значение.")

    t1 = threading.Thread(target=transaction_a)
    t2 = threading.Thread(target=transaction_b)
    t1.start(); t2.start()
    t1.join(); t2.join()
    val = coll.find_one({"_id": 1})
    print(f"[Main] Чтение данных: balance = {val['balance']}")

In [6]:
if __name__ == "__main__":
  demo_dirty_read()


--- Проверка Dirty Read (Грязное чтение) ---
[A] Начало транзакции. Меняю balance на 500...
[A] Чтение данных: balance = 500
[B] Чтение данных: balance = 100
[B] УСПЕХ: Грязное чтение невозможно. База вернула старое значение.
[A] Отмена транзакции (Abort)...
[Main] Чтение данных: balance = 100


In [7]:
def demo_non_repeatable_read():
    print("\n--- Проверка Non-repeatable Read (Неповторяющееся чтение) ---")
    
    def transaction_a():
        with client.start_session() as session:
            with session.start_transaction():
                val1 = coll.find_one({"_id": 1}, session=session)
                print(f"[A] Первое чтение: balance = {val1['balance']}")
                
                time.sleep(3)
                
                val2 = coll.find_one({"_id": 1}, session=session)
                print(f"[A] Второе чтение: balance = {val2['balance']}")
                
                if val1['balance'] == val2['balance']:
                    print("[A] УСПЕХ: Чтение идентичное (Snapshot Isolation работает).")
                else:
                    print("[A] ОШИБКА: Данные изменились внутри транзакции!")

    def transaction_b():
        time.sleep(1)
        coll.update_one({"_id": 1}, {"$set": {"balance": 999}})
        print("[B] Изменяю balance на 999 и фиксирую (Commit)...")
        val = coll.find_one({"_id": 1})
        print(f"[B] Чтение: balance = {val['balance']}")

    t1 = threading.Thread(target=transaction_a)
    t2 = threading.Thread(target=transaction_b)
    t1.start(); t2.start()
    t1.join(); t2.join()
    val = coll.find_one({"_id": 1})
    print(f"[Main] Чтение: balance = {val['balance']}")

In [8]:
if __name__ == "__main__":
  demo_non_repeatable_read()


--- Проверка Non-repeatable Read (Неповторяющееся чтение) ---
[A] Первое чтение: balance = 100
[B] Изменяю balance на 999 и фиксирую (Commit)...
[B] Чтение: balance = 999
[A] Второе чтение: balance = 100
[A] УСПЕХ: Чтение идентичное (Snapshot Isolation работает).
[Main] Чтение: balance = 999


In [9]:
def demo_phantom_read():
    print("\n--- Проверка Phantom Read (Чтение-фантом) ---")
    coll.drop()
    coll.insert_many([{"_id": 1, "type": "task"}, {"_id": 2, "type": "task"}])

    def transaction_a():
        with client.start_session() as session:
            with session.start_transaction():
                count1 = coll.count_documents({"type": "task"}, session=session)
                print(f"[A] Первое чтение: найдено {count1} задач")
                
                time.sleep(3)
                
                count2 = coll.count_documents({"type": "task"}, session=session)
                print(f"[A] Второе чтение: найдено {count2} задач")
                
                if count1 == count2:
                    print("[A] РЕЗУЛЬТАТ: Фантомов нет. Snapshot Isolation защитил от аномалии.")
                else:
                    print("[A] РЕЗУЛЬТАТ: Обнаружен Phantom Read!")

    def transaction_b():
        time.sleep(1)
        print("[B] Вставляю новую задачу (фантомный документ)...")
        coll.insert_one({"_id": 3, "type": "task"})
        print("[B] Документ вставлен.")
        count = coll.count_documents({"type": "task"})
        print(f"[B] Чтение: найдено {count} задач")

    t1 = threading.Thread(target=transaction_a)
    t2 = threading.Thread(target=transaction_b)
    t1.start(); t2.start()
    t1.join(); t2.join()
    count = coll.count_documents({"type": "task"})
    print(f"[Main] Чтение: найдено {count} задач")

In [10]:
if __name__ == "__main__":
  demo_phantom_read()


--- Проверка Phantom Read (Чтение-фантом) ---
[A] Первое чтение: найдено 2 задач
[B] Вставляю новую задачу (фантомный документ)...
[B] Документ вставлен.
[B] Чтение: найдено 3 задач
[A] Второе чтение: найдено 2 задач
[A] РЕЗУЛЬТАТ: Фантомов нет. Snapshot Isolation защитил от аномалии.
[Main] Чтение: найдено 3 задач


In [11]:
def demo_write_skew():
    print("\n--- Проверка Write Skew (Перекос записи) ---")
    # Сценарий: В больнице должно дежурить минимум 1 врач.
    # Сейчас дежурят двое: Алиса и Боб.
    coll.drop()
    coll.insert_many([
        {"name": "Alice", "onCall": True},
        {"name": "Bob", "onCall": True}
    ])

    def doctor_quits(name):
        with client.start_session() as session:
            try:
                with session.start_transaction():
                    on_call_count = coll.count_documents({"onCall": True}, session=session)
                    print(f"[{name}] Видит врачей на дежурстве: {on_call_count}")
                    
                    if on_call_count >= 2:
                        print(f"[{name}] Решает уйти с дежурства...")
                        time.sleep(2)
                        coll.update_one({"name": name}, {"$set": {"onCall": False}}, session=session)
                        print(f"[{name}] Успешно ушел.")
                    else:
                        print(f"[{name}] Не может уйти, остался последний врач!")
            except Exception as e:
                print(f"[{name}] Ошибка: {e}")

    t1 = threading.Thread(target=doctor_quits, args=("Alice",))
    t2 = threading.Thread(target=doctor_quits, args=("Bob",))
    
    t1.start(); t2.start()
    t1.join(); t2.join()

    final_count = coll.count_documents({"onCall": True})
    print(f"\nИТОГ: Врачей на дежурстве осталось: {final_count}")
    if final_count == 0:
        print("РЕЗУЛЬТАТ: Write Skew воспроизведен! Правило нарушено, база позволила обоим уйти.")
    else:
        print("РЕЗУЛЬТАТ: Write Skew не произошел.")

In [12]:
if __name__ == "__main__":
  demo_write_skew()


--- Проверка Write Skew (Перекос записи) ---
[Alice] Видит врачей на дежурстве: 2
[Alice] Решает уйти с дежурства...
[Bob] Видит врачей на дежурстве: 2
[Bob] Решает уйти с дежурства...
[Alice] Успешно ушел.
[Bob] Успешно ушел.

ИТОГ: Врачей на дежурстве осталось: 0
РЕЗУЛЬТАТ: Write Skew воспроизведен! Правило нарушено, база позволила обоим уйти.


In [13]:

def demo_write_skew_fixed():
    print("\n--- Проверка Write Skew (Перекос записи) ---")
    # Сценарий: В больнице должно дежурить минимум 1 врач.
    # Сейчас дежурят двое: Алиса и Боб.
    coll.drop()
    coll.insert_many([
        {"name": "Alice", "onCall": True},
        {"name": "Bob", "onCall": True}
    ])
    coll.insert_one({"_id": "lock", "type": "on_call_counter"})
    def doctor_quits_fixed(name):
        with client.start_session() as session:
            try:
                with session.start_transaction():
                    coll.update_one({"_id": "lock"}, {"$set": {"last_updated_by": name}}, session=session)
                    
                    on_call_count = coll.count_documents({"onCall": True}, session=session)
                    print(f"[{name}] Видит врачей: {on_call_count}")

                    if on_call_count >= 2:
                        coll.update_one({"name": name}, {"$set": {"onCall": False}}, session=session)
                        print(f"[{name}] Успешно ушел.")
                    else:
                        print(f"[{name}] Остановка: нельзя оставить больницу пустой!")
            except errors.OperationFailure:
                print(f"[{name}] Конфликт записи. Транзакция будет перезапущена драйвером.")
    t1 = threading.Thread(target=doctor_quits_fixed, args=("Alice",))
    t2 = threading.Thread(target=doctor_quits_fixed, args=("Bob",))
    
    t1.start(); t2.start()
    t1.join(); t2.join()

    final_count = coll.count_documents({"onCall": True})
    print(f"\nИТОГ: Врачей на дежурстве осталось: {final_count}")
    if final_count == 0:
        print("РЕЗУЛЬТАТ: Write Skew воспроизведен! Правило нарушено, база позволила обоим уйти.")
    else:
        print("РЕЗУЛЬТАТ: Write Skew не произошел.")

In [14]:
if __name__ == "__main__":
  demo_write_skew_fixed()


--- Проверка Write Skew (Перекос записи) ---
[Alice] Видит врачей: 2
[Alice] Успешно ушел.
[Bob] Конфликт записи. Транзакция будет перезапущена драйвером.

ИТОГ: Врачей на дежурстве осталось: 1
РЕЗУЛЬТАТ: Write Skew не произошел.


# Отчет

## 1. Выбор СУБД и среда развертывания
Для выполнения работы выбрана NoSQL СУБД MongoDB.
Среда запуска: Docker-контейнер, развернутый в режиме Replica Set (необходимо для поддержки многодокументных транзакций).

### Команды запуска:
```bash
docker run -d --name mongo-lab -p 27017:27017 mongo:latest --replSet rs0
docker exec -it mongo-lab mongosh --eval 'rs.initiate({_id: "rs0", members: [{_id: 0, host: "localhost:27017"}]})'
```

## 2. Уровни изоляции и аномалии (Анализ по стандарту SQL)
В ходе исследования установлено, что в MongoDB стандартные уровни изоляции SQL (Read Uncommitted, Read Committed, Repeatable Read, Serializable) отсутствуют как явные настройки. Вместо них для транзакций используется уровень Snapshot Isolation (Изоляция на основе снимков).

### Результаты тестирования аномалий внутри транзакции:
| Аномалия | Результат теста | Техническое обоснование |
| :---: | :---: | :---: |
| Dirty Read | Не воспроизводится | Использование движка WiredTiger и MVCC гарантирует, что изменения не видны другим сессиям до commit |
| Non-repeatable Read | Не воспроизводится | Транзакция фиксирует логическое время (Snapshot), обеспечивая консистентность чтений одного и того же документа |
| Phantom Read | Не воспроизводится | Снимок данных (Snapshot) исключает появление новых документов в результатах выборок внутри одной транзакции |
| Write Skew | Воспроизводится | Единственная аномалия, допустимая в Snapshot Isolation. Возникает при логическом конфликте между изменениями разных документов |

## 4. Анализ MongoDB с точки зрения CAP-теоремы
Согласно официальной документации и архитектурным особенностям, MongoDB позиционируется как CP-система (Consistency + Partition Tolerance).

* Partition Tolerance (P): Как распределенная система, MongoDB спроектирована для работы в условиях сетевых разделений. Использование Replica Set позволяет системе выживать при потере связи между узлами.

* Consistency (C): При возникновении сетевого разрыва (Split-brain), узел Primary, оказавшийся в меньшинстве, прекращает прием записей. Это гарантирует, что в системе не возникнет конфликтующих версий данных.

* Availability (A) — Жертва: В момент перевыборов Primary (от 2 до 12 секунд) или при отсутствии кворума, база данных становится недоступной на запись. Таким образом, MongoDB жертвует стопроцентной доступностью ради обеспечения строгой согласованности данных.

## 5. Вывод
В ходе лабораторной работы было выявлено, что MongoDB обеспечивает высокий уровень изоляции данных (Snapshot Isolation) для многодокументных транзакций. Это автоматически защищает разработчика от большинства классических аномалий, описанных в стандарте SQL. Однако для предотвращения специфических логических аномалий (Write Skew) требуется либо дополнительная логика на уровне приложения (блокировки), либо использование атомарных операций над одним документом.